# Explosive NSE moves — master research notebook

Rare-event prediction: classify **upcoming explosive moves** (forward MFE ≥ threshold). No separate 5d/10d/20d speed tiers.

**Causal rule:** features at market close **T**; labels from **T+1…T+H** only.

**Order:** Phase 1 audit → panel → events → **pre-event discovery before ML** → features → ablation → walk-forward ML → strategy → scanner.

Artifacts: `results/explosive_moves/`. Override data path with `QUANT_DATA_DIR`.

**Limitations (v1):** no split-adjusted prices; sector/VIX/free-float only when datasets are added.


**Resume:** each phase skips heavy work when parquet/csv artifacts and `results/explosive_moves/meta/*_meta.json` fingerprints match. Force rebuild: `CFG["rerun_all"] = True` or `CFG["force_rerun"] = ["panel", "events", ...]` (see cell 4).


In [ ]:
import pickle
from quant.research.explosive_research import (
    trading_calendar_from_panel,
    calendar_shift,
    mark_ca_suspect,
    forward_excursion_calendar,
    ca_invalidates_forward_window,
    first_touch_triple_barrier,
    dedupe_event_episodes,
    daily_precision_lift,
    summarize_daily_metrics,
    fingerprint_dict,
    force_rerun_phase,
    phase_cache_hit,
    write_phase_meta,
    read_meta_fingerprint,
    label_cfg_fingerprint,
    file_fingerprint,
    LABEL_ENGINE_VERSION,
    label_symbol_panel,
    purge_overlaps_calendar,
)

from pathlib import Path
import json, warnings, os, sys, gc, subprocess, importlib.util
warnings.filterwarnings("ignore")
if importlib.util.find_spec("tqdm") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tqdm"])
from tqdm.auto import tqdm


from quant.config import DATA_DIR as QUANT_DATA_DIR, PARQUET_DATA_DIR, NIFTY50_PATH

DATA_DIR = PARQUET_DATA_DIR
OUT_DIR = QUANT_DATA_DIR / "results" / "explosive_moves"
for sub in tqdm(("panel", "events", "features", "models", "reports", "reports/figures", "scanner"), desc="Setup — output dirs"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

PANEL_PATH = OUT_DIR / "panel" / "ohlcv_panel.parquet"
EVENTS_PATH = OUT_DIR / "events" / "explosive_events.parquet"
FEATURES_PATH = OUT_DIR / "features" / "feature_dataset.parquet"
AUDIT_PATH = OUT_DIR / "reports" / "data_audit.json"
PRE_EVENT_PATH = OUT_DIR / "reports" / "pre_event_summary.parquet"
ABLATION_PATH = OUT_DIR / "reports" / "feature_ablation.csv"
MODEL_EVAL_PATH = OUT_DIR / "reports" / "model_evaluation.csv"
SCANNER_PATH = OUT_DIR / "scanner" / "latest_ranked.parquet"
SCANNER_CSV = OUT_DIR / "scanner" / "latest_ranked.csv"

assert DATA_DIR.exists(), f"Missing parquet dir: {DATA_DIR}"
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)
print("NIFTY:", NIFTY50_PATH, "exists" if NIFTY50_PATH.exists() else "missing")

META_DIR = OUT_DIR / "meta"
META_DIR.mkdir(parents=True, exist_ok=True)
PANEL_META_PATH = OUT_DIR / "panel" / "panel_build_meta.json"
PANEL_FEATURES_PATH = OUT_DIR / "features" / "panel_features.parquet"
MODEL_BUNDLE_PATH = OUT_DIR / "models" / "primary_model_bundle.pkl"
ABLATION_MODEL_PATH = OUT_DIR / "models" / "ablation_logit.pkl"
AUDIT_META_PATH = META_DIR / "audit_meta.json"
EVENTS_META_PATH = META_DIR / "events_meta.json"
FEATURES_META_PATH = META_DIR / "features_meta.json"
PRE_EVENT_META_PATH = META_DIR / "pre_event_meta.json"
ABLATION_META_PATH = META_DIR / "ablation_meta.json"
ML_META_PATH = META_DIR / "ml_meta.json"
SIM_META_PATH = META_DIR / "sim_meta.json"
SCANNER_META_PATH = META_DIR / "scanner_meta.json"

models = {}
calibrator = None
full_model = None
full_cols = []


def load_model_bundle(path=MODEL_BUNDLE_PATH):
    if not path.exists():
        return None
    return pickle.loads(path.read_bytes())


def hydrate_models_from_bundle():
    """Load LGBM + isotonic from disk (resume after ML skip)."""
    global models, calibrator, Xcols
    bundle = load_model_bundle()
    if bundle is None:
        return False
    models = {"lgbm": bundle["model"]}
    calibrator = bundle.get("calibrator")
    if bundle.get("feature_cols"):
        Xcols = bundle["feature_cols"]
    return True


In [ ]:
required = ["pandas", "numpy", "pyarrow", "duckdb", "matplotlib", "scipy", "tqdm"]
missing = [x for x in required if importlib.util.find_spec(x) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb
import matplotlib.pyplot as plt
from scipy import stats
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.options.mode.copy_on_write = True

if importlib.util.find_spec("sklearn") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV

HAS_LGBM = importlib.util.find_spec("lightgbm") is not None
if not HAS_LGBM:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
        HAS_LGBM = True
    except Exception:
        HAS_LGBM = False
if HAS_LGBM:
    import lightgbm as lgb

print("duckdb:", duckdb.__version__, "| lightgbm:", HAS_LGBM)


## Phase 0 — Research configuration


In [ ]:
with tqdm(total=1, desc="Phase 0 — config") as pbar:
    CFG = {
        # Forward window for MFE measurement (not a "must move within N days" speed rule).
        "primary_horizon": 20,
        "h_max": 20,
        "horizons": [20],
        "primary_mfe_thresh": 2.0,
        "triple_barrier": {"up": 0.20, "down": -0.10},
        "min_price": 20.0,
        "min_history_bars": 60,
        "min_median_turnover_20d": 1_000_000.0,
        "eq_series_only": True,
        "top_n_events": 500,
        "pre_event_lags": [5, 10, 20],
        "event_episode_gap": 10,
        "match_k": 3,
        "random_seed": 42,
        "ca_gap_abs_exclude": 0.35,
        "wf_splits": {
            "train_end": 2018,
            "valid_end": 2020,
            "test1_end": 2022,
            "test2_end": 2024,
        },
        "costs": {
            "slippage_bps": 20.0,
            "brokerage_bps": 0.0,
            "stt_bps": 10.0,
            "exchange_bps": 3.25,
            "stamp_bps": 1.5,
            "sebi_bps": 0.0001,
            "gst_rate": 0.18,
        },
        "strategy": {
            "top_k": (10, 20),
            "prob_threshold": 0.25,
            "max_hold": 20,
            "stop_loss": 0.10,
            "max_participation_pct": 0.05,
            "initial_capital": 1_000_000.0,
            "max_missing_sessions": 10,
        },
        # Drop symbols with patchy vendor history (see nse_data_inspection coverage).
        "symbol_continuity": {
            "enabled": True,
            "min_coverage_ratio": 0.95,
            "max_internal_gap_sessions": 20,
        },
        # Resume: skip phases when artifacts + meta fingerprint match. Override:
        #   CFG["rerun_all"] = True  OR  CFG["force_rerun"] = ["panel", "events", ...]
        "rerun_all": False,
        "force_rerun": [],
        "parallel_workers": 0,  # 0 = auto (min(4, cpu_count)); 1 = serial symbols
        "ml": {
            # Walk-forward: train only these on each split (rf is slow on full panel)
            "walk_forward_models": ["lgbm"],
            "benchmark_models": ["logit", "rf"],
            "benchmark_splits": ["valid"],
            "lgbm_n_estimators": 200,
            "rf_n_estimators": 80,
            "train_sample_max": 400_000,
        },
    }

    FEATURE_FLAGS = {
        "HAS_DELIVERY": False,
        "HAS_TURNOVER": False,
        "HAS_SERIES": False,
        "HAS_PREV_CLOSE": False,
    }

    print(json.dumps(CFG, indent=2))
    pbar.update(1)


## Phase 1 — Data audit


In [ ]:
def stock_parquet_glob():
    return sorted(
        p for p in tqdm(
            DATA_DIR.rglob("nse_cm_*.parquet"),
            desc="Phase 1 — discover files",
            unit="path",
        )
        if "results" not in p.parts and "indices" not in p.parts
    )

files = stock_parquet_glob()
print("Daily stock files:", len(files))
if not files:
    raise FileNotFoundError(f"No nse_cm_*.parquet under {DATA_DIR}")

_audit_fp = fingerprint_dict({"n_files": len(files)})

if phase_cache_hit(CFG, "audit", [AUDIT_PATH], AUDIT_META_PATH, _audit_fp):
    audit = json.loads(AUDIT_PATH.read_text())
    FEATURE_FLAGS.update(audit.get("feature_flags", {}))
    audit_df = pd.DataFrame(audit.get("duckdb_summary", []))
    display(audit_df)
    if audit.get("duplicate_sample"):
        print("Duplicate (date,symbol) rows (sample):", len(audit["duplicate_sample"]))
    print("Feature flags:", FEATURE_FLAGS)
else:
    schema_union = set()
    for p in tqdm([files[0], files[len(files) // 2], files[-1]], desc="Phase 1 — schema samples"):
        schema_union |= {c.lower() for c in pq.ParquetFile(p).schema.names}

    con = duckdb.connect()
    paths = [str(p).replace("'", "''") for p in files]
    paths_sql = ", ".join(f"'{p}'" for p in paths)
    with tqdm(total=2, desc="Phase 1 — DuckDB audit") as pbar:
        audit_df = con.execute(
            f'''
            SELECT COUNT(*) AS n_rows,
                   COUNT(DISTINCT symbol) AS n_symbols,
                   MIN(CAST(date AS DATE)) AS dmin,
                   MAX(CAST(date AS DATE)) AS dmax
            FROM read_parquet([{paths_sql}], union_by_name=true)
            '''
        ).df()
        pbar.update(1)
        dup_df = con.execute(
            f'''
            SELECT date, symbol, COUNT(*) AS n
            FROM read_parquet([{paths_sql}], union_by_name=true)
            GROUP BY 1,2 HAVING n>1 LIMIT 10
            '''
        ).df()
        pbar.update(1)

    FEATURE_FLAGS = {
        "HAS_TURNOVER": "turnover" in schema_union,
        "HAS_DELIVERY": ("deliv_per" in schema_union) or ("deliv_qty" in schema_union),
        "HAS_SERIES": "series" in schema_union,
        "HAS_PREV_CLOSE": "prev_close" in schema_union,
    }
    audit = {
        "n_files": len(files),
        "schema_union": sorted(schema_union),
        "feature_flags": FEATURE_FLAGS,
        "duckdb_summary": audit_df.to_dict(orient="records"),
        "duplicate_sample": dup_df.to_dict(orient="records"),
        "nifty_exists": NIFTY50_PATH.exists(),
    }
    AUDIT_PATH.write_text(json.dumps(audit, indent=2, default=str))
    write_phase_meta(AUDIT_META_PATH, "audit", _audit_fp)
    display(audit_df)
    if not dup_df.empty:
        print("Duplicate (date,symbol) rows (sample):", len(dup_df))
    print("Feature flags:", FEATURE_FLAGS)


## Phase 2 — Panel construction


In [ ]:
def first_series(frame, name):
    s = frame[name]
    return s.iloc[:, 0] if isinstance(s, pd.DataFrame) else s


def normalize_ohlc(df, source_name):
    cols = {str(c).lower().strip(): c for c in df.columns}
    aliases = {
        "date": ["date", "datetime", "timestamp"],
        "symbol": ["symbol", "ticker", "tradingsymbol"],
        "open": ["open"],
        "high": ["high"],
        "low": ["low"],
        "close": ["close", "adj close", "adj_close"],
        "volume": ["volume", "vol", "qty", "quantity", "total traded quantity"],
        "turnover": ["turnover", "tottrdval", "traded_value"],
        "deliv_qty": ["deliv_qty", "deliverablequantity", "delivqty"],
        "deliv_per": ["deliv_per", "delivper", "delivery_percentage"],
        "series": ["series"],
        "prev_close": ["prev_close", "prevclose", "previous_close"],
    }
    out = {}
    for target, names in aliases.items():
        found = next((cols[n] for n in names if n in cols), None)
        if found is not None:
            out[target] = found
    missing = [x for x in ["date", "high", "low", "close"] if x not in out]
    if missing:
        raise ValueError(f"{source_name}: missing {missing}")
    symbol = first_series(df, out["symbol"]) if "symbol" in out else Path(source_name).stem
    x = pd.DataFrame({
        "Date": pd.to_datetime(first_series(df, out["date"]), errors="coerce"),
        "Symbol": pd.Series(symbol, index=df.index).astype(str).str.upper().str.strip(),
        "Open": pd.to_numeric(first_series(df, out["open"]), errors="coerce") if "open" in out else np.nan,
        "High": pd.to_numeric(first_series(df, out["high"]), errors="coerce"),
        "Low": pd.to_numeric(first_series(df, out["low"]), errors="coerce"),
        "Close": pd.to_numeric(first_series(df, out["close"]), errors="coerce"),
        "Volume": pd.to_numeric(first_series(df, out["volume"]), errors="coerce") if "volume" in out else np.nan,
    })
    opt_map = {
        "turnover": "Turnover",
        "deliv_qty": "DelivQty",
        "deliv_per": "DelivPer",
        "prev_close": "PrevClose",
    }
    for src, dst in opt_map.items():
        if src in out:
            x[dst] = pd.to_numeric(first_series(df, out[src]), errors="coerce")
    if "series" in out:
        x["Series"] = first_series(df, out["series"]).astype(str).str.upper().str.strip()
    x = x.dropna(subset=["Date", "Close"]).loc[x["Close"] > 0]
    if getattr(x["Date"].dt, "tz", None) is not None:
        x["Date"] = x["Date"].dt.tz_convert("UTC").dt.tz_localize(None)
    for c in x.select_dtypes(include=["float64"]).columns:
        x[c] = x[c].astype("float32")
    x["Symbol"] = x["Symbol"].astype("string")
    return x


PANEL_EXTRA = ["Turnover", "DelivQty", "DelivPer", "Series", "PrevClose"]
BASE_COLS = ["Date", "Symbol", "Open", "High", "Low", "Close", "Volume"]


PANEL_META_PATH = OUT_DIR / "panel" / "panel_build_meta.json"


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


PANEL_META_PATH = OUT_DIR / "panel" / "panel_build_meta.json"


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


PANEL_META_PATH = OUT_DIR / "panel" / "panel_build_meta.json"


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


PANEL_META_PATH = OUT_DIR / "panel" / "panel_build_meta.json"


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


PANEL_META_PATH = OUT_DIR / "panel" / "panel_build_meta.json"


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


def rebuild_panel(force=False):
    files = stock_parquet_glob()
    fp = panel_source_fingerprint(files)
    meta = {}
    if PANEL_META_PATH.exists():
        meta = json.loads(PANEL_META_PATH.read_text())
    force = force or force_rerun_phase(CFG, "panel")
    if PANEL_PATH.exists() and not force and meta.get("fingerprint") == fp:
        print("Using existing panel (fingerprint match):", PANEL_PATH)
        return
    writer = None
    n_ok = n_err = 0
    for i, f in enumerate(tqdm(files, desc="Phase 2 — stitch panel", unit="file"), 1):
        try:
            x = normalize_ohlc(pd.read_parquet(f), f.name)
            keep = [c for c in BASE_COLS + PANEL_EXTRA if c in x.columns]
            table = pa.Table.from_pandas(x[keep], preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(PANEL_PATH, table.schema, compression="zstd")
            writer.write_table(table)
            n_ok += 1
            if i % 400 == 0:
                gc.collect()
        except Exception as exc:
            n_err += 1
            if n_err <= 5:
                print("skip", f.name, exc)
    if writer:
        writer.close()
    PANEL_META_PATH.write_text(json.dumps({"fingerprint": fp, "n_files": len(files), "n_ok": n_ok, "n_err": n_err}, indent=2))
    print("Panel OK:", n_ok, "errors:", n_err)


with tqdm(total=2, desc="Phase 2 — panel load") as pbar:
    rebuild_panel(force=False)
    pbar.update(1)
    panel = pd.read_parquet(PANEL_PATH)
    panel["Date"] = pd.to_datetime(panel["Date"])
    panel["Symbol"] = panel["Symbol"].astype("string")
    panel = panel.sort_values(["Symbol", "Date"])
    n_dup = int(panel.duplicated(["Date", "Symbol"]).sum())
    if n_dup:
        print("Dropping duplicate Date-Symbol rows:", n_dup)
        panel = panel.drop_duplicates(["Date", "Symbol"], keep="last")
    panel = panel.reset_index(drop=True)

    if "Turnover" not in panel.columns:
        panel["Turnover"] = (panel["Close"] * panel["Volume"]).astype("float32")

    if CFG["eq_series_only"] and "Series" in panel.columns:
        bad = panel["Series"].notna() & ~panel["Series"].str.upper().eq("EQ")
        panel = panel.loc[~bad]

    from quant.research.explosive_research import apply_symbol_continuity_filter

    nifty = None
    if NIFTY50_PATH.exists():
        nifty = normalize_ohlc(pd.read_parquet(NIFTY50_PATH), "NIFTY50")
        nifty = nifty.sort_values("Date")
        nifty["nifty_ret_1"] = nifty["Close"].pct_change()

    pre_rows, pre_syms = len(panel), panel["Symbol"].nunique()
    panel, continuity_excluded = apply_symbol_continuity_filter(panel, nifty, CFG)
    if not continuity_excluded.empty:
        rep = OUT_DIR / "reports" / "symbol_continuity_excluded.parquet"
        rep.parent.mkdir(parents=True, exist_ok=True)
        continuity_excluded.to_parquet(rep, index=False)
        print(
            "Symbol continuity filter:",
            len(continuity_excluded),
            "symbols excluded,",
            pre_rows - len(panel),
            "rows removed (",
            f"{100 * (pre_rows - len(panel)) / max(pre_rows, 1):.1f}% ),",
            "cfg:",
            CFG.get("symbol_continuity"),
        )

    print(panel.shape, panel["Date"].min(), "→", panel["Date"].max(), "symbols:", panel["Symbol"].nunique())
    pbar.update(1)


## Phase 3 — Event labeling (MFE / MAE / forward close return)


In [ ]:
import os
from concurrent.futures import ProcessPoolExecutor, as_completed


def explosive_mfe_threshold(cfg):
    return float(cfg["primary_mfe_thresh"])


def add_labels(df, cfg):
    df = df.copy()
    H = int(cfg["primary_horizon"])
    thr = explosive_mfe_threshold(cfg)
    mfe = df[f"mfe_{H}"]
    ok = mfe.notna() & df[f"horizon_complete_{H}"] & ~df[f"ca_fwd_invalid_{H}"]
    df[f"y_explosive_{H}"] = ((mfe >= thr) & ok).astype("int8")
    return df


TRADING_CAL = trading_calendar_from_panel(panel)
_events_fp = fingerprint_dict(
    {
        "panel": read_meta_fingerprint(PANEL_META_PATH),
        "labels": label_cfg_fingerprint(CFG),
        "symbol_continuity": CFG.get("symbol_continuity"),
    }
)

if phase_cache_hit(CFG, "events", [EVENTS_PATH], EVENTS_META_PATH, _events_fp):
    events = pd.read_parquet(EVENTS_PATH)
    print("Loaded events:", len(events), EVENTS_PATH)
else:
    sym_groups = [(sym, g) for sym, g in panel.groupby("Symbol", observed=True)]
    min_len = CFG["min_history_bars"] + CFG["h_max"] + 1
    sym_groups = [(s, g) for s, g in sym_groups if len(g) >= min_len]

    tb = CFG["triple_barrier"]
    horizons = CFG["horizons"]
    ca_gap = CFG["ca_gap_abs_exclude"]
    workers = int(CFG.get("parallel_workers", 0))
    if workers <= 0:
        workers = min(4, os.cpu_count() or 1)

    def _label_one(pair):
        _, gsym = pair
        return label_symbol_panel(
            gsym, TRADING_CAL, horizons, ca_gap_abs=ca_gap, tb_up=tb["up"], tb_down=tb["down"]
        )

    labeled_parts = []
    if workers > 1 and len(sym_groups) > workers * 2:
        with ProcessPoolExecutor(max_workers=workers) as pool:
            futs = [pool.submit(_label_one, p) for p in sym_groups]
            for fut in tqdm(as_completed(futs), total=len(futs), desc="Phase 3 — label symbols", unit="sym"):
                labeled_parts.append(fut.result())
    else:
        for pair in tqdm(sym_groups, desc="Phase 3 — label symbols", unit="sym"):
            labeled_parts.append(_label_one(pair))

    with tqdm(total=2, desc="Phase 3 — finalize events") as bar:
        labeled = pd.concat(labeled_parts, ignore_index=True)
        bar.update(1)
        labeled = add_labels(labeled, CFG)
        bar.update(1)

    H = CFG["primary_horizon"]
    labeled["ca_excluded"] = labeled["ca_suspect"].astype("int8")

    print("Explosive label: MFE >=", explosive_mfe_threshold(CFG), f"within {H} forward sessions (window only)")
    print("Incomplete horizons:", int((~labeled[f"horizon_complete_{CFG['h_max']}"]).sum()))
    print("CA-forward invalid:", int(labeled[f"ca_fwd_invalid_{CFG['h_max']}"].sum()))

    events = labeled.dropna(subset=[f"mfe_{CFG['primary_horizon']}"])
    events.to_parquet(EVENTS_PATH, compression="zstd", index=False)
    write_phase_meta(EVENTS_META_PATH, "events", _events_fp, n_rows=len(events))
    print("Wrote", EVENTS_PATH, len(events))


## Phase 5 — Feature dataset (stock-day rows)


In [ ]:
from quant.research.explosive_research import feature_column_names


def compute_panel_features(panel_df, nifty_df=None):
    """Rolling features on the full tradable panel (labels merged later)."""
    x = panel_df.sort_values(["Symbol", "Date"]).copy()
    x["Date"] = pd.to_datetime(x["Date"], errors="coerce")
    g = x.groupby("Symbol", observed=True)

    x["vol_rvol_20"] = x["Volume"] / g["Volume"].transform(lambda s: s.rolling(20, min_periods=10).mean())
    x["vol_va"] = g["Volume"].transform(lambda s: s.rolling(5, min_periods=3).mean()) / g["Volume"].transform(
        lambda s: s.rolling(20, min_periods=10).mean()
    )
    x["vol_turn_accel"] = g["Turnover"].transform(lambda s: s.rolling(5, min_periods=3).mean()) / g["Turnover"].transform(
        lambda s: s.rolling(20, min_periods=10).mean()
    )

    prev_c = g["Close"].shift(1)
    tr = np.maximum(x["High"] - x["Low"], np.maximum((x["High"] - prev_c).abs(), (x["Low"] - prev_c).abs()))
    x["vola_atr14_pct"] = tr.groupby(x["Symbol"]).transform(lambda s: s.rolling(14, min_periods=5).mean()) / x["Close"] * 100
    x["vola_atr5"] = tr.groupby(x["Symbol"]).transform(lambda s: s.rolling(5, min_periods=3).mean())
    x["vola_atr50"] = tr.groupby(x["Symbol"]).transform(lambda s: s.rolling(50, min_periods=20).mean())
    x["vola_vr"] = x["vola_atr5"] / x["vola_atr50"]
    roll_std = g["Close"].transform(lambda s: s.rolling(20, min_periods=10).std())
    x["vola_bbw"] = (4 * roll_std) / g["Close"].transform(lambda s: s.rolling(20, min_periods=10).mean())

    x["px_ret_5"] = g["Close"].pct_change(5)
    x["px_ret_10"] = g["Close"].pct_change(10)
    x["px_ret_20"] = g["Close"].pct_change(20)
    x["px_dist_20h"] = x["Close"] / g["High"].transform(lambda s: s.rolling(20, min_periods=10).max()) - 1
    x["px_dist_50h"] = x["Close"] / g["High"].transform(lambda s: s.rolling(50, min_periods=20).max()) - 1
    x["px_dist_252h"] = x["Close"] / g["High"].transform(lambda s: s.rolling(252, min_periods=60).max()) - 1
    x["px_clv"] = (x["Close"] - x["Low"]) / (x["High"] - x["Low"]).replace(0, np.nan)
    up = (x["Close"] > g["Close"].shift(1)).astype(int)
    x["px_up_streak"] = up.groupby(x["Symbol"]).transform(lambda s: s.groupby((s != s.shift()).cumsum()).cumsum())

    if "DelivPer" in x.columns:
        x["deliv_ma5"] = g["DelivPer"].transform(lambda s: s.rolling(5, min_periods=3).mean())
        x["deliv_ma20"] = g["DelivPer"].transform(lambda s: s.rolling(20, min_periods=10).mean())
        x["deliv_accel"] = x["deliv_ma5"] / x["deliv_ma20"]
    if "DelivQty" in x.columns:
        x["deliv_qty_accel"] = g["DelivQty"].transform(lambda s: s.rolling(5, min_periods=3).mean()) / g["DelivQty"].transform(
            lambda s: s.rolling(20, min_periods=10).mean()
        )

    x["liq_turn_med20"] = g["Turnover"].transform(lambda s: s.rolling(20, min_periods=10).median())
    daily_ret = g["Close"].pct_change()
    x["liq_zero_ret_20"] = daily_ret.groupby(x["Symbol"]).transform(
        lambda s: (s.abs() < 1e-8).rolling(20, min_periods=10).mean()
    )
    if "PrevClose" in x.columns:
        x["liq_gap"] = x["Close"] / x["PrevClose"] - 1

    if nifty_df is not None:
        nd = nifty_df[["Date", "Close"]].copy()
        nd["Date"] = pd.to_datetime(nd["Date"], errors="coerce")
        nd = nd.sort_values("Date").drop_duplicates("Date", keep="last")
        nd = nd.assign(
            mkt_nifty_ret_5=nd["Close"].pct_change(5),
            mkt_nifty_ret_20=nd["Close"].pct_change(20),
        )
        gs = x.groupby("Symbol", observed=True)
        sma20 = gs["Close"].transform(lambda s: s.rolling(20, min_periods=10).mean())
        breadth = (
            x.assign(_above=(x["Close"] > sma20).astype(int))
            .groupby("Date", observed=True)["_above"]
            .mean()
            .rename("mkt_breadth_20sma")
        )
        x = x.merge(nd[["Date", "mkt_nifty_ret_5", "mkt_nifty_ret_20"]], on="Date", how="left")
        x["rs_20"] = x["px_ret_20"] - x["mkt_nifty_ret_20"]
        x = x.merge(breadth, on="Date", how="left")
    return x


def attach_event_labels(feat_df, events_df):
    label_cols = [c for c in events_df.columns if c.startswith(("mfe_", "mae_", "ret_", "y_", "horizon_", "ca_"))]
    out = feat_df.merge(events_df[["Date", "Symbol"] + label_cols], on=["Date", "Symbol"], how="left")
    out = out.loc[out["Close"] >= CFG["min_price"]]
    out = out.loc[out["liq_turn_med20"] >= CFG["min_median_turnover_20d"]]
    return out


_features_fp = fingerprint_dict(
    {
        "engine": LABEL_ENGINE_VERSION,
        "panel": read_meta_fingerprint(PANEL_META_PATH),
        "events": read_meta_fingerprint(EVENTS_META_PATH),
        "min_price": CFG["min_price"],
        "min_turnover": CFG["min_median_turnover_20d"],
        "nifty": file_fingerprint(NIFTY50_PATH),
        "symbol_continuity": CFG.get("symbol_continuity"),
    }
)

if phase_cache_hit(CFG, "features", [FEATURES_PATH, PANEL_FEATURES_PATH], FEATURES_META_PATH, _features_fp):
    panel_features = pd.read_parquet(PANEL_FEATURES_PATH)
    features = pd.read_parquet(FEATURES_PATH)
    feature_cols = feature_column_names(panel_features.columns)
    label_cols = [c for c in features.columns if c.startswith("y_")]
    print("Loaded features:", panel_features.shape, features.shape)
else:
    with tqdm(total=3, desc="Phase 5 — feature dataset") as pbar:
        panel_features = compute_panel_features(panel, nifty)
        pbar.update(1)
        feature_cols = feature_column_names(panel_features.columns)
        features = attach_event_labels(panel_features, events)
        pbar.update(1)
        label_cols = [c for c in features.columns if c.startswith("y_")]
        print("panel_features:", panel_features.shape, "| labeled rows:", len(features), "features:", len(feature_cols))
        features.to_parquet(FEATURES_PATH, compression="zstd", index=False)
        panel_features.to_parquet(PANEL_FEATURES_PATH, compression="zstd", index=False)
        write_phase_meta(FEATURES_META_PATH, "features", _features_fp, n_features=len(feature_cols))
        pbar.update(1)
    print("Wrote", FEATURES_PATH, "and", PANEL_FEATURES_PATH)


## Phase 4 — Pre-event discovery (after Phase 5)


In [ ]:
_pre_fp = fingerprint_dict(
    {
        "features": read_meta_fingerprint(FEATURES_META_PATH),
        "events": read_meta_fingerprint(EVENTS_META_PATH),
        "lags": CFG["pre_event_lags"],
        "top_n_events": CFG["top_n_events"],
        "event_episode_gap": CFG["event_episode_gap"],
        "match_k": CFG["match_k"],
        "random_seed": CFG["random_seed"],
        "primary_mfe_thresh": CFG["primary_mfe_thresh"],
    }
)
_cohort_path = OUT_DIR / "reports" / "pre_event_cohort.parquet"

if phase_cache_hit(CFG, "pre_event", [PRE_EVENT_PATH, _cohort_path], PRE_EVENT_META_PATH, _pre_fp):
    pre_tests = pd.read_parquet(PRE_EVENT_PATH)
    print("Loaded pre-event:", len(pre_tests), PRE_EVENT_PATH)
    display(pre_tests.head(20))
else:
    rng = np.random.default_rng(CFG["random_seed"])
    H = CFG["primary_horizon"]
    mfe_col, y_col = f"mfe_{H}", f"y_explosive_{H}"

    event_pool = events.loc[
        events[mfe_col].notna()
        & events[f"horizon_complete_{H}"]
        & ~events[f"ca_fwd_invalid_{H}"]
        & (events[mfe_col] >= CFG["primary_mfe_thresh"])
    ].copy()

    candidates = dedupe_event_episodes(
        event_pool,
        mfe_col=mfe_col,
        calendar=TRADING_CAL,
        gap_sessions=CFG["event_episode_gap"],
    ).sort_values(mfe_col, ascending=False).head(CFG["top_n_events"])

    if candidates.empty:
        raise ValueError("No explosive events after dedup; check panel / thresholds.")

    def liquidity_bucket(turnover):
        if not np.isfinite(turnover) or turnover <= 0:
            return "unk"
        return ["micro", "small", "mid", "large"][min(3, int(np.log10(turnover + 1) // 2))]

    candidates["liq_bucket"] = candidates["Turnover"].map(liquidity_bucket)
    pool = events.loc[events["Close"] >= CFG["min_price"]]
    used_controls: set[tuple] = set()
    LAGS = CFG["pre_event_lags"]
    FEATURES_PRE = [c for c in feature_cols if c in panel_features.columns]
    feat_index = panel_features.set_index(["Date", "Symbol"])

    def match_controls(evt_row, k=3):
        same_day = pool.loc[pool["Date"] == evt_row["Date"]]
        same_day = same_day.loc[same_day[y_col] == 0]
        if same_day.empty:
            return pd.DataFrame()
        lo, hi = evt_row["Close"] * 0.5, evt_row["Close"] * 2.0
        same_day = same_day.loc[(same_day["Close"] >= lo) & (same_day["Close"] <= hi)]
        b = evt_row["liq_bucket"]
        same_day = same_day.loc[same_day["Turnover"].map(liquidity_bucket) == b]
        if same_day.empty:
            same_day = pool.loc[(pool["Date"] == evt_row["Date"]) & (pool[y_col] == 0)]
        same_day = same_day.loc[~same_day.apply(lambda r: (r["Date"], r["Symbol"]) in used_controls, axis=1)]
        if len(same_day) <= k:
            pick = same_day
        else:
            pick = same_day.sample(n=k, random_state=rng.integers(0, 2**31 - 1))
        for _, r in pick.iterrows():
            used_controls.add((r["Date"], r["Symbol"]))
        return pick

    cohort_rows = []
    for _, ev in tqdm(candidates.iterrows(), total=len(candidates), desc="Phase 4 — match controls"):
        ctl = match_controls(ev, CFG["match_k"])
        if ctl.empty:
            continue
        for lag in LAGS:
            ev_asof = calendar_shift(TRADING_CAL, pd.Series([ev["Date"]]), -lag).iloc[0]
            if pd.isna(ev_asof):
                continue
            key = (pd.Timestamp(ev_asof), ev["Symbol"])
            if key not in feat_index.index:
                continue
            ev_rec = {
                "cohort": "event",
                "lag": lag,
                "event_date": ev["Date"],
                "asof_date": ev_asof,
                "Symbol": ev["Symbol"],
            }
            for c in FEATURES_PRE:
                ev_rec[c] = feat_index.loc[key, c]
            cohort_rows.append(ev_rec)
            for _, cr in ctl.iterrows():
                ctl_asof = calendar_shift(TRADING_CAL, pd.Series([cr["Date"]]), -lag).iloc[0]
                if pd.isna(ctl_asof):
                    continue
                ckey = (pd.Timestamp(ctl_asof), cr["Symbol"])
                if ckey not in feat_index.index:
                    continue
                ctl_rec = {
                    "cohort": "control",
                    "lag": lag,
                    "event_date": ev["Date"],
                    "asof_date": ctl_asof,
                    "Symbol": cr["Symbol"],
                }
                for c in FEATURES_PRE:
                    ctl_rec[c] = feat_index.loc[ckey, c]
                cohort_rows.append(ctl_rec)

    cohort = pd.DataFrame(cohort_rows)
    if cohort.empty:
        raise ValueError("Pre-event cohort empty; check panel_features.")

    n_tests = len(FEATURES_PRE) * len(LAGS)
    tests = []
    for lag, fc in tqdm(
        [(lag, fc) for lag in LAGS for fc in FEATURES_PRE],
        desc="Phase 4 — Mann-Whitney",
        leave=False,
    ):
        ev_s = pd.to_numeric(cohort.loc[(cohort["cohort"] == "event") & (cohort["lag"] == lag), fc], errors="coerce")
        ctl_s = pd.to_numeric(cohort.loc[(cohort["cohort"] == "control") & (cohort["lag"] == lag), fc], errors="coerce")
        a = ev_s.to_numpy(dtype=np.float64)
        b = ctl_s.to_numpy(dtype=np.float64)
        a, b = a[np.isfinite(a)], b[np.isfinite(b)]
        if len(a) < 30 or len(b) < 30:
            continue
        stat, p = stats.mannwhitneyu(a, b, alternative="two-sided")
        tests.append({
            "lag": lag,
            "feature": fc,
            "n_events": len(a),
            "n_controls": len(b),
            "event_median": float(np.median(a)),
            "control_median": float(np.median(b)),
            "u_stat": float(stat),
            "p_value": float(p),
            "p_bonferroni": min(1.0, float(p) * n_tests),
        })

    pre_tests = pd.DataFrame(tests).sort_values("p_bonferroni")
    pre_tests.to_parquet(PRE_EVENT_PATH, compression="zstd", index=False)
    cohort.to_parquet(_cohort_path, compression="zstd", index=False)
    write_phase_meta(PRE_EVENT_META_PATH, "pre_event", _pre_fp, n_tests=len(pre_tests))
    print("Deduped events:", len(candidates), "| cohort rows:", len(cohort), "| tests:", len(pre_tests))
    display(pre_tests.head(20))


## Phase 6 — Controlled experiments (feature-family ablation)


In [ ]:
from quant.research.explosive_research import load_feature_dataset_artifacts

if not (globals().get("feature_cols") and globals().get("features") is not None):
    _feat_ctx = load_feature_dataset_artifacts(FEATURES_PATH, PANEL_FEATURES_PATH)
    globals().update(_feat_ctx)
    print(
        "Hydrated Phase 5 from disk:",
        len(_feat_ctx["feature_cols"]),
        "features,",
        len(_feat_ctx["features"]),
        "rows",
    )

FAMILIES = {
    "E1_volume": [c for c in feature_cols if c.startswith("vol_")],
    "E2_volatility": [c for c in feature_cols if c.startswith("vola_")],
    "E3_delivery": [c for c in feature_cols if c.startswith("deliv_")],
    "E4_price_structure": [c for c in feature_cols if c.startswith("px_")],
    "E5_relative_strength": [c for c in feature_cols if c.startswith("rs_")],
    "E6_liquidity": [c for c in feature_cols if c.startswith("liq_")],
    "E7_market": [c for c in feature_cols if c.startswith("mkt_")],
}

TARGET = f"y_explosive_{CFG['primary_horizon']}"


def split_masks(df):
    y = df["Date"].dt.year
    s = CFG["wf_splits"]
    train = y <= s["train_end"]
    valid = (y > s["train_end"]) & (y <= s["valid_end"])
    test1 = (y > s["valid_end"]) & (y <= s["test1_end"])
    test2 = (y > s["test1_end"]) & (y <= s["test2_end"])
    holdout = y > s["test2_end"]
    return train, valid, test1, test2, holdout


def precision_at_k(y_true, y_score, k=20):
    if len(y_true) == 0:
        return np.nan
    k = min(k, len(y_true))
    idx = np.argsort(-y_score)[:k]
    return float(y_true.iloc[idx].mean())


def lift_at_k(y_true, y_score, k=20):
    base = y_true.mean()
    if base <= 0:
        return np.nan
    return precision_at_k(y_true, y_score, k) / base


def fit_logit_scorer(train_df, cols, target):
    cols = [c for c in cols if c in train_df.columns]
    if not cols:
        return None, cols
    tr = train_df.dropna(subset=cols + [target])
    if tr[target].sum() < 5:
        return None, cols
    X = tr[cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    y = tr[target]
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    w = len(y) / max(y.sum(), 1)
    clf = LogisticRegression(max_iter=500, class_weight={0: 1, 1: w})
    clf.fit(Xs, y)
    return (scaler, clf), cols


def score_df(df, model, cols, target):
    if model is None:
        return pd.Series(np.zeros(len(df)), index=df.index)
    scaler, clf = model
    X = df[cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    return pd.Series(clf.predict_proba(scaler.transform(X))[:, 1], index=df.index)


def eval_daily_scanner(frame, scores, target, k=20):
    tmp = frame[["Date", target]].copy()
    tmp["score"] = scores
    daily = daily_precision_lift(tmp, date_col="Date", score_col="score", y_col=target, k=k)
    return daily, summarize_daily_metrics(daily, k)


train_m, valid_m, test1_m, test2_m, holdout_m = split_masks(features)

_ab_fp = fingerprint_dict(
    {"features": read_meta_fingerprint(FEATURES_META_PATH), "wf_splits": CFG["wf_splits"], "target": TARGET}
)

if phase_cache_hit(CFG, "ablation", [ABLATION_PATH], ABLATION_META_PATH, _ab_fp):
    ablation = pd.read_csv(ABLATION_PATH)
    if ABLATION_MODEL_PATH.exists():
        full_model, full_cols = pickle.loads(ABLATION_MODEL_PATH.read_bytes())
    else:
        full_model, full_cols = None, feature_cols
    display(ablation)
else:
    train_ab = features.loc[train_m]
    valid_ab = features.loc[valid_m]
    all_cols = feature_cols

    full_model, full_cols = fit_logit_scorer(train_ab, all_cols, TARGET)
    scores = score_df(valid_ab, full_model, full_cols, TARGET)
    _, full_daily = eval_daily_scanner(valid_ab, scores, TARGET, k=20)

    rows = []
    for fam, cols in tqdm(FAMILIES.items(), desc="Phase 6 — ablation"):
        ablate_cols = [c for c in all_cols if c not in cols]
        m, used = fit_logit_scorer(train_ab, ablate_cols, TARGET)
        sc = score_df(valid_ab, m, used, TARGET)
        _, daily = eval_daily_scanner(valid_ab, sc, TARGET, k=20)
        rows.append({
            "experiment": fam,
            "mean_precision_at_20": daily["mean_precision_at_20"],
            "mean_lift_at_20": daily["mean_lift_at_20"],
            "pct_days_with_hit": daily["pct_days_with_hit"],
            "delta_lift_vs_full": daily["mean_lift_at_20"] - full_daily["mean_lift_at_20"],
            "n_features_removed": len(cols),
        })
    rows.append({
        "experiment": "full",
        "mean_precision_at_20": full_daily["mean_precision_at_20"],
        "mean_lift_at_20": full_daily["mean_lift_at_20"],
        "pct_days_with_hit": full_daily["pct_days_with_hit"],
        "delta_lift_vs_full": 0.0,
        "n_features_removed": 0,
    })
    ablation = pd.DataFrame(rows)
    ablation.to_csv(ABLATION_PATH, index=False)
    if full_model is not None:
        ABLATION_MODEL_PATH.write_bytes(pickle.dumps((full_model, full_cols)))
    write_phase_meta(ABLATION_META_PATH, "ablation", _ab_fp)
    display(ablation)


## Phase 7 — ML + purged walk-forward


In [ ]:
from sklearn.isotonic import IsotonicRegression
from quant.research.explosive_research import load_feature_dataset_artifacts

if not (globals().get("feature_cols") and globals().get("features") is not None):
    _feat_ctx = load_feature_dataset_artifacts(FEATURES_PATH, PANEL_FEATURES_PATH)
    globals().update(_feat_ctx)
    print(
        "Hydrated Phase 5 from disk:",
        len(_feat_ctx["feature_cols"]),
        "features,",
        len(_feat_ctx["features"]),
        "rows",
    )

MIN_TRAIN_POS = 20
Xcols = [c for c in feature_cols if c in features.columns]
TARGET = f"y_explosive_{CFG['primary_horizon']}"
ML_CFG = CFG.get("ml", {})


def _subsample_train(X, y, f, max_rows):
    if not max_rows or len(X) <= max_rows:
        return X, y, f
    rng = np.random.default_rng(CFG["random_seed"])
    if y.sum() < 5:
        idx = rng.choice(len(X), size=max_rows, replace=False)
    else:
        pos_idx = np.flatnonzero(y.to_numpy() == 1)
        neg_idx = np.flatnonzero(y.to_numpy() == 0)
        n_pos = min(len(pos_idx), max(5, max_rows // 10))
        n_neg = max_rows - n_pos
        pick = np.concatenate([
            rng.choice(pos_idx, size=n_pos, replace=False),
            rng.choice(neg_idx, size=min(n_neg, len(neg_idx)), replace=False),
        ])
        idx = np.sort(pick)
    return X.iloc[idx], y.iloc[idx], f.iloc[idx]


_ml_fp = fingerprint_dict(
    {
        "features": read_meta_fingerprint(FEATURES_META_PATH),
        "wf_splits": CFG["wf_splits"],
        "h_max": CFG["h_max"],
        "random_seed": CFG["random_seed"],
        "ml": ML_CFG,
    }
)

if phase_cache_hit(CFG, "ml", [MODEL_EVAL_PATH, MODEL_BUNDLE_PATH], ML_META_PATH, _ml_fp):
    hydrate_models_from_bundle()
    model_eval = pd.read_csv(MODEL_EVAL_PATH)
    display(model_eval)
else:

    def prep_xy(frame):
        f = frame.dropna(subset=Xcols + [TARGET])
        X = f[Xcols].replace([np.inf, -np.inf], np.nan).fillna(0)
        y = f[TARGET]
        return X, y, f

    def eval_probs_daily(frame, scores, tag, k=20):
        y = frame[TARGET]
        _, summ = eval_daily_scanner(frame, scores, TARGET, k=k)
        out = {"split": tag, "pr_auc": float(average_precision_score(y, scores)) if y.sum() > 0 else np.nan}
        out.update(summ)
        out["brier"] = float(brier_score_loss(y, scores))
        out["base_rate"] = float(y.mean())
        out["n"] = int(len(y))
        return out

    wf_models = list(ML_CFG.get("walk_forward_models", ["lgbm"]))
    bench_models = list(ML_CFG.get("benchmark_models", ["logit", "rf"]))
    bench_splits = set(ML_CFG.get("benchmark_splits", ["valid"]))
    train_cap = ML_CFG.get("train_sample_max")
    lgbm_trees = int(ML_CFG.get("lgbm_n_estimators", 200))
    rf_trees = int(ML_CFG.get("rf_n_estimators", 80))

    results = []
    calibrator = None

    tr_mask = purge_overlaps_calendar(
        train_m, features["Date"], features.loc[valid_m, "Date"].min(), CFG["h_max"], TRADING_CAL
    )
    Xtr, ytr, _ = prep_xy(features.loc[tr_mask])
    Xva, yva, fva = prep_xy(features.loc[valid_m])

    if ytr.sum() >= MIN_TRAIN_POS and len(Xva) > 50 and HAS_LGBM and "lgbm" in wf_models:
        with tqdm(total=2, desc="Phase 7 — primary LGBM + isotonic") as pbar:
            scale = (ytr == 0).sum() / max((ytr == 1).sum(), 1)
            primary_lgbm = lgb.LGBMClassifier(
                n_estimators=lgbm_trees,
                learning_rate=0.05,
                num_leaves=63,
                subsample=0.8,
                colsample_bytree=0.8,
                class_weight={0: 1.0, 1: scale},
                random_state=CFG["random_seed"],
                n_jobs=-1,
            )
            primary_lgbm.fit(Xtr, ytr)
            pbar.update(1)
            p_va_raw = primary_lgbm.predict_proba(Xva)[:, 1]
            calibrator = IsotonicRegression(out_of_bounds="clip")
            calibrator.fit(p_va_raw, yva)
            models["lgbm"] = primary_lgbm
            pbar.update(1)

    split_iter = [("valid", valid_m), ("test1", test1_m), ("test2", test2_m), ("holdout", holdout_m)]
    for name, mask in tqdm(split_iter, desc="Phase 7 — walk-forward eval"):
        Xte, yte, fte = prep_xy(features.loc[mask])
        if len(Xte) < 50:
            continue

        run_bench = name in bench_splits
        models_this = list(wf_models)
        if run_bench:
            for m in bench_models:
                if m not in models_this:
                    models_this.append(m)

        if "logit" in models_this or "rf" in models_this:
            tr_mask = train_m
            if name != "valid":
                tr_mask = train_m | valid_m
            test_start = features.loc[mask, "Date"].min()
            if pd.notna(test_start):
                tr_mask = purge_overlaps_calendar(tr_mask, features["Date"], test_start, CFG["h_max"], TRADING_CAL)
            Xtr, ytr, ftr = prep_xy(features.loc[tr_mask])
            if ytr.sum() < MIN_TRAIN_POS or len(Xtr) < 100:
                print(f"skip benchmarks {name}: insufficient train")
            else:
                Xtr_b, ytr_b, _ = _subsample_train(Xtr, ytr, ftr, train_cap)
                if "logit" in models_this:
                    scaler = StandardScaler()
                    Xtr_s = scaler.fit_transform(Xtr_b)
                    Xte_s = scaler.transform(Xte)
                    logit = LogisticRegression(max_iter=400, class_weight="balanced")
                    logit.fit(Xtr_s, ytr_b)
                    p_logit = logit.predict_proba(Xte_s)[:, 1]
                    results.append(eval_probs_daily(fte, pd.Series(p_logit, index=fte.index), f"logit_{name}"))
                if "rf" in models_this:
                    rf = RandomForestClassifier(
                        n_estimators=rf_trees,
                        max_depth=8,
                        min_samples_leaf=50,
                        class_weight="balanced",
                        random_state=CFG["random_seed"],
                        n_jobs=-1,
                    )
                    rf.fit(Xtr_b, ytr_b)
                    p_rf = rf.predict_proba(Xte)[:, 1]
                    results.append(eval_probs_daily(fte, pd.Series(p_rf, index=fte.index), f"rf_{name}"))

        if "lgbm" in models_this and models.get("lgbm") is not None and calibrator is not None:
            p_raw = models["lgbm"].predict_proba(Xte)[:, 1]
            p_lgb = calibrator.predict(p_raw)
            results.append(eval_probs_daily(fte, pd.Series(p_lgb, index=fte.index), f"lgbm_{name}"))

    if models.get("lgbm") is not None and calibrator is not None:
        bundle = {
            "model": models["lgbm"],
            "calibrator": calibrator,
            "feature_cols": Xcols,
            "target": TARGET,
            "cfg_snippet": {
                "primary_horizon": CFG["primary_horizon"],
                "prob_threshold": CFG["strategy"]["prob_threshold"],
            },
        }
        MODEL_BUNDLE_PATH.write_bytes(pickle.dumps(bundle))
        models["lgbm"].booster_.save_model(str(OUT_DIR / "models" / "lgbm_primary.txt"))
        print("Saved bundle:", MODEL_BUNDLE_PATH)

    model_eval = pd.DataFrame(results)
    model_eval.to_csv(MODEL_EVAL_PATH, index=False)
    write_phase_meta(ML_META_PATH, "ml", _ml_fp)
    display(model_eval)


## Phase 8 — Executable strategy simulation (research)


In [ ]:
from quant.research.explosive_research import (
    SIM_ENGINE_VERSION,
    report_equity_diagnostics,
    simulate_topk_calendar,
    trades_to_round_trips,
)

TRADES_ALL_CSV = OUT_DIR / "reports" / "trades_all.csv"


def write_round_trip_csv(trade_log: pd.DataFrame, path: Path) -> None:
    if trade_log.empty:
        return
    trades_to_round_trips(trade_log).to_csv(path, index=False)


def export_all_trades_csv() -> None:
    parts = []
    for k in CFG["strategy"]["top_k"]:
        trade_path = OUT_DIR / "reports" / f"trades_top{k}.parquet"
        if not trade_path.exists():
            continue
        rt = trades_to_round_trips(pd.read_parquet(trade_path))
        if rt.empty:
            continue
        rt = rt.assign(top_k=k)
        parts.append(rt)
    if not parts:
        return
    out = pd.concat(parts, ignore_index=True)
    TRADES_ALL_CSV.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(TRADES_ALL_CSV, index=False)
    print("Wrote", TRADES_ALL_CSV, f"({len(out)} round trips)")


eq_paths = [OUT_DIR / "reports" / f"equity_top{k}.parquet" for k in CFG["strategy"]["top_k"]]
_sim_fp = fingerprint_dict(
    {
        "ml": read_meta_fingerprint(ML_META_PATH),
        "top_k": CFG["strategy"]["top_k"],
        "strategy": CFG["strategy"],
        "sim_engine": SIM_ENGINE_VERSION,
    }
)

if phase_cache_hit(CFG, "sim", eq_paths, SIM_META_PATH, _sim_fp):
    if not models.get("lgbm"):
        hydrate_models_from_bundle()
    for k in CFG["strategy"]["top_k"]:
        sp = OUT_DIR / "reports" / f"equity_top{k}_stats.json"
        if sp.exists():
            print(f"Top-{k} (cached):", json.loads(sp.read_text()))
elif not models.get("lgbm") and not hydrate_models_from_bundle():
    print("Calibrated LightGBM unavailable; skip strategy simulation.")
else:
    test = features.loc[test2_m | holdout_m].copy()
    Xte = test[Xcols].replace([np.inf, -np.inf], np.nan).fillna(0)
    test_scores = calibrator.predict(models["lgbm"].predict_proba(Xte)[:, 1])
    for k in tqdm(CFG["strategy"]["top_k"], desc="Phase 8 — backtest"):
        eq, stout, trades = simulate_topk_calendar(test, panel, test_scores, k, CFG, TRADING_CAL)
        if not eq.empty:
            print(f"Top-{k} sim:", {kk: (f"{vv:.2%}" if kk == "total_return" else vv) for kk, vv in stout.items()})
            eq.to_parquet(OUT_DIR / "reports" / f"equity_top{k}.parquet", index=False)
            Path(OUT_DIR / "reports" / f"equity_top{k}_stats.json").write_text(json.dumps(stout, indent=2))
            if not trades.empty:
                trades.to_parquet(OUT_DIR / "reports" / f"trades_top{k}.parquet", index=False)
                write_round_trip_csv(trades, OUT_DIR / "reports" / f"trades_top{k}.csv")
    write_phase_meta(SIM_META_PATH, "sim", _sim_fp)

export_all_trades_csv()

print("\n--- Equity diagnostics (cached or fresh) ---")
for k in CFG["strategy"]["top_k"]:
    path = OUT_DIR / "reports" / f"equity_top{k}.parquet"
    if path.exists():
        report_equity_diagnostics(
            pd.read_parquet(path),
            f"TOP {k}",
            CFG["strategy"]["initial_capital"],
        )
    trade_path = OUT_DIR / "reports" / f"trades_top{k}.parquet"
    if trade_path.exists():
        tl = pd.read_parquet(trade_path)
        sells = tl.loc[tl["side"] == "SELL"]
        print(f"\nTOP {k} trade log: {len(tl)} rows ({len(sells)} exits)")
        if not sells.empty and "notional" in sells.columns:
            print("Exit reasons:", sells["reason"].value_counts().to_dict())


## Phase 9 — Daily scanner export


In [ ]:
_scan_fp = fingerprint_dict(
    {
        "panel_features": file_fingerprint(PANEL_FEATURES_PATH),
        "ml": read_meta_fingerprint(ML_META_PATH),
        "prob_threshold": CFG["strategy"]["prob_threshold"],
    }
)

if phase_cache_hit(CFG, "scanner", [SCANNER_PATH, SCANNER_CSV], SCANNER_META_PATH, _scan_fp):
    scan = pd.read_parquet(SCANNER_PATH)
    rank_cols = ["Date", "Symbol", "Close", "prob_explosive", "vol_rvol_20", "vola_vr", "px_dist_252h", "rs_20"]
    rank_cols = [c for c in rank_cols if c in scan.columns]
    print("Loaded scanner:", scan["Date"].max() if "Date" in scan.columns else "?", len(scan), SCANNER_PATH)
    display(scan[rank_cols].head(20))
else:
    if not models.get("lgbm"):
        hydrate_models_from_bundle()

    with tqdm(total=4, desc="Phase 9 — scanner") as pbar:
        latest_date = panel_features["Date"].max()
        scan = panel_features.loc[panel_features["Date"] == latest_date].copy()
        pbar.update(1)

        prob_thresh = CFG["strategy"]["prob_threshold"]
        if "lgbm" in models and calibrator is not None:
            X = scan[Xcols].replace([np.inf, -np.inf], np.nan).fillna(0)
            scan["prob_explosive"] = calibrator.predict(models["lgbm"].predict_proba(X)[:, 1])
        elif full_model is not None:
            scan["prob_explosive"] = score_df(scan, full_model, full_cols, TARGET)
        else:
            scan["prob_explosive"] = np.nan
        pbar.update(1)

        if np.isfinite(prob_thresh):
            scan = scan.loc[scan["prob_explosive"] >= prob_thresh]
        pbar.update(1)

        rank_cols = ["Date", "Symbol", "Close", "prob_explosive", "vol_rvol_20", "vola_vr", "px_dist_252h", "rs_20"]
        rank_cols = [c for c in rank_cols if c in scan.columns]
        scan = scan.sort_values("prob_explosive", ascending=False)
        scan[rank_cols].head(50).to_csv(SCANNER_CSV, index=False)
        scan.to_parquet(SCANNER_PATH, compression="zstd", index=False)
        write_phase_meta(
            SCANNER_META_PATH,
            "scanner",
            _scan_fp,
            asof_date=str(latest_date),
        )
        pbar.update(1)

    print("Scanner date", latest_date, "rows:", len(scan), "→", SCANNER_PATH)
    display(scan[rank_cols].head(20))
